In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
from bm_preproc import BoyerMoore, boyer_moore

In [14]:
# Approximate matching using the pigeonhole principle and Boyer-Moore as
# the exact-match primitive. Implementation follows Ben Langmead's lecture
# in Course 3 of the JHU "Algorithms for DNA Sequencing" specialization.
#
# Modification from the lecture version: an explicit `exceeded` flag tracks
# whether a candidate has been rejected, allowing the suffix verification
# loop to be skipped entirely when the prefix already exceeds the budget.
# The lecture version still enters the suffix loop and breaks out on the
# first comparison; this version avoids that wasted entry.
def approximate_match(p, t, n):
    
    # Each segment is roughly len(p)/(n+1); round handles non-integer cases
    segment_size = round(len(p) / (n+1))
    
    # Use a set so duplicate hits from multiple segments collapse to one entry
    all_matches = set()
    
    # Iterate over each of the n+1 segments
    for i in range(n+1):
        
        start = i * segment_size
        
        # Cap the segment at len(p) so the last segment doesn't overshoot
        end = min(start + segment_size, len(p))
        
        # Preprocess THIS segment as a standalone pattern for Boyer-Moore
        p_bm = BoyerMoore(p[start:end], alphabet='ACGT')
        
        # Find every exact match of the segment in t — these are seed candidates
        matches = boyer_moore(p[start:end], p_bm, t)
        
        # Verify each seed candidate by counting mismatches across the full p
        for m in matches:
            
            # Reject candidates whose alignment runs off either end of t
            if m - start < 0 or m - start + len(p) > len(t):
                continue
            
            mismatches = 0
            exceeded = False
            
            # Verify the prefix of p (positions before the segment)
            for j in range(0, start):
                if not p[j] == t[m - start + j]:
                    mismatches += 1
                    if mismatches > n:
                        exceeded = True
                        break
            
            # Skip the suffix verification entirely if prefix already failed
            if not exceeded:
                for j in range(end, len(p)):
                    if not p[j] == t[m - start + j]:
                        mismatches += 1
                        if mismatches > n:
                            exceeded = True
                            break
            
            # Record the implied start of full p in t — not m, which is segment offset
            if not exceeded:
                all_matches.add(m - start)
    
    return list(all_matches)

In [16]:
p = 'AACTAGCTTA'
t = 'GAGCAAACTAGCTAAGAGCC'

print(approximate_match(p, t, 0))    # expect []   — 1 mismatch exceeds budget
print(approximate_match(p, t, 1))    # expect [5]  — 1 mismatch within budget
print(approximate_match(p, t, 2))    # expect [5]  — still within budget

[]
[5]
[5]
